In [1]:
import requests

BASE = "https://openlibrary.org"
headers = {
    "User-Agent": "MyAppName/1.0 (myemail@example.com)"
}

def _get_json(url: str):
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    return r.json()

def _normalize_description(desc):
    """
    Handles both:
      - "description": "text..."
      - "description": {"type": "/type/text", "value": "text..."}
    """
    if not desc:
        return None
    if isinstance(desc, str):
        return desc.strip() or None
    if isinstance(desc, dict):
        value = desc.get("value")
        if isinstance(value, str):
            return value.strip() or None
    return None

def get_openlibrary_description_by_isbn(isbn: str):
    # 1. Get edition JSON for this ISBN
    edition_url = f"{BASE}/isbn/{isbn}.json"
    try:
        edition = _get_json(edition_url)
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:
            return None
        raise

    # 2. Try edition-level description
    desc = _normalize_description(edition.get("description"))
    if desc:
        return desc

    # 3. Follow linked work(s)
    works = edition.get("works") or []
    if not works:
        return None

    work_key = works[0].get("key")  # e.g. "/works/OL59048W"
    if not work_key:
        return None

    work_url = f"{BASE}{work_key}.json"
    work = _get_json(work_url)

    # 4. Work-level description
    return _normalize_description(work.get("description"))


if __name__ == "__main__":
    # Example: one ISBN for The Remains of the Day
    # (Replace with the exact ISBN you are using)
    isbn = "9780582342989"  # example; use your target ISBN
    #print(get_openlibrary_description_by_isbn(isbn))
    print(_get_json(f"{BASE}/isbn/{isbn}.json"))

{'publishers': ['Longman'], 'identifiers': {'goodreads': ['829059'], 'librarything': ['3035']}, 'weight': '2.9 ounces', 'first_sentence': 'Tonight, I find myself here in a guest house in the city of Salisbury.', 'series': ['Penguin Reader , Level 6  (3000 words)'], 'covers': [381418], 'physical_format': 'Paperback', 'key': '/books/OL7880640M', 'authors': [{'key': '/authors/OL28493A'}], 'languages': [{'key': '/languages/eng'}], 'source_records': ['bwb:9780582342989', 'ia:remainsofday0000rice'], 'title': 'The Remains of the Day', 'notes': 'Penguin Joint Venture Readers', 'number_of_pages': 112, 'edition_name': 'New Ed edition', 'subjects': ['English language readers'], 'publish_date': 'August 14, 2000', 'works': [{'key': '/works/OL59048W'}], 'type': {'key': '/type/edition'}, 'physical_dimensions': '7.6 x 4.9 x 0.2 inches', 'ocaid': 'remainsofday0000rice', 'isbn_10': ['0582342988'], 'isbn_13': ['9780582342989'], 'lc_classifications': ['PR6068.I12145 R46 2000'], 'latest_revision': 12, 'rev

In [3]:
print(get_openlibrary_description_by_isbn("9780593395561"))

Ryland Grace is the sole survivor on a desperate, last-chance mission–and if he fails, humanity and the earth itself will perish. Except that right now, he doesn’t know that. He can’t even remember his own name, let alone the nature of his assignment or how to complete it. All he knows is that he’s been asleep for a very, very long time. And he’s just been awakened to find himself millions of miles from home, with nothing but two corpses for company.

His crewmates dead, his memories fuzzily returning, he realizes that an impossible task now confronts him. Alone on this tiny ship that’s been cobbled together by every government and space agency on the planet and hurled into the depths of space, it’s up to him to conquer an extinction-level threat to our species.

And thanks to an unexpected ally, he just might have a chance.

Part scientific mystery, part dazzling interstellar journey, *Project Hail Mary* is a tale of discovery, speculation, and survival to rival *The Martian*–while ta

In [4]:
url = "https://openlibrary.org/works/OL59048W.json?fields=key,title,author_name,isbn,first_publish_year,subject"
print(_get_json(url))

{'description': "In the summer of 1956, Stevens, the ageing butler of Darlington Hall, embarks on a leisurely holiday that will take him deep into the countryside and into his past . . .A contemporary classic, The Remains of the Day is Kazuo Ishiguro's beautiful and haunting evocation of life between the wars in a Great English House, of lost causes and lost love.", 'title': 'The Remains of the Day', 'covers': [95742, 420075, 3224016, 9002755, 381418, 8665333, 979075, 9425849, 10795405, 10795406, 11116153, 1013800, 11690971, 12274255, 12331371], 'subject_places': ['England'], 'first_publish_date': '1994', 'key': '/works/OL59048W', 'authors': [{'author': {'key': '/authors/OL28493A'}, 'type': {'key': '/type/author_role'}}], 'type': {'key': '/type/work'}, 'subjects': ['Man Booker Prize Winner', 'award:man_booker_prize=1989', 'Literature', 'Household employees', 'Butlers', 'Country homes', 'Domestics', 'Fiction', 'Country homes -- Fiction', 'Man-woman relationships', 'History', 'Historical

# Book List Scraping

In [10]:
import requests
from bs4 import BeautifulSoup
import time
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ISBN-Scraper/1.0)"
}

def get_book_urls_from_list(list_url, limit=10):
    book_urls = []
    page = 1

    while len(book_urls) < limit:
        url = f"{list_url}?page={page}"
        r = requests.get(url, headers=HEADERS)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "html.parser")

        for a in soup.select("a.bookTitle"):
            book_urls.append("https://www.goodreads.com" + a["href"])
            if len(book_urls) == limit:
                return book_urls

        page += 1

    return book_urls

def extract_work_id(book_url):
    r = requests.get(book_url, headers=HEADERS)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # Look for editions link
    for a in soup.select('a[href^="/work/editions/"]'):
        match = re.search(r"/work/editions/(\d+)", a["href"])
        if match:
            return match.group(1)

    return None
    
def extract_isbn13_from_work(work_id, max_editions=5):
    url = f"https://www.goodreads.com/work/editions/{work_id}"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    isbn13_list = []

    # ISBNs are in edition rows
    for row in soup.select("div.editionData"):
        text = row.get_text(" ", strip=True)

        match = re.search(r"\b97[89]\d{10}\b", text)
        if match:
            isbn13_list.append(match.group())

        if len(isbn13_list) == max_editions:
            break

    return isbn13_list

def extract_isbn13_from_list(list_url, limit=10):
    results = []

    book_urls = get_book_urls_from_list(list_url, limit)

    for book_url in book_urls:
        print(f"Processing: {book_url}")

        work_id = extract_work_id(book_url)
        if not work_id:
            results.append({
                "book_url": book_url,
                "work_id": None,
                "isbn13": None,
            })
            continue

        isbn13s = extract_isbn13_from_work(work_id)

        results.append({
            "book_url": book_url,
            "work_id": work_id,
            "isbn13": isbn13s,
        })

        time.sleep(1)  # be polite

    return results


In [11]:
if __name__ == "__main__":
    list_url = "https://www.goodreads.com/list/show/952.1001_Books_You_Must_Read_Before_You_Die"
    data = extract_isbn13_from_list(list_url, limit=10)


Processing: https://www.goodreads.com/book/show/2657.To_Kill_a_Mockingbird
Processing: https://www.goodreads.com/book/show/1885.Pride_and_Prejudice
Processing: https://www.goodreads.com/book/show/40961427-1984
Processing: https://www.goodreads.com/book/show/33.The_Lord_of_the_Rings
Processing: https://www.goodreads.com/book/show/4671.The_Great_Gatsby
Processing: https://www.goodreads.com/book/show/10210.Jane_Eyre
Processing: https://www.goodreads.com/book/show/170448.Animal_Farm
Processing: https://www.goodreads.com/book/show/5907.The_Hobbit_or_There_and_Back_Again
Processing: https://www.goodreads.com/book/show/157993.The_Little_Prince
Processing: https://www.goodreads.com/book/show/5107.The_Catcher_in_the_Rye


In [12]:
print(data)

[{'book_url': 'https://www.goodreads.com/book/show/2657.To_Kill_a_Mockingbird', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/1885.Pride_and_Prejudice', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/40961427-1984', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/33.The_Lord_of_the_Rings', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/4671.The_Great_Gatsby', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/10210.Jane_Eyre', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/170448.Animal_Farm', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/5907.The_Hobbit_or_There_and_Back_Again', 'work_id': None, 'isbn13': None}, {'book_url': 'https://www.goodreads.com/book/show/157993.The_Little_Prince', 'work_id': None, 'isbn13': None}, {'book_url':

### Reading dumps data

In [4]:
import duckdb
import json

con = duckdb.connect('open_library.db')
file_path = 'data/ol_dump_works_2026-03-31.txt.gz'

# Step 1: Load the raw data first
con.execute(f"""
    CREATE OR REPLACE TABLE works_raw AS
    SELECT *
    FROM read_csv(
        '{file_path}',
        header=False,
        delim='\t',
        columns={{
            't': 'VARCHAR',
            'k': 'VARCHAR',
            'r': 'INTEGER',
            'd': 'VARCHAR',
            'json_data': 'JSON'
        }}
    )
""")

# Step 2: Inspect the JSON keys from a sample row to build the schema
sample = con.execute("SELECT json_data FROM works_raw LIMIT 1").fetchone()
if sample:
    keys = list(json.loads(sample[0]).keys())
    print("Detected JSON keys:", keys)

# Step 3: Create the works table by unpacking JSON fields
# DuckDB can extract JSON fields dynamically like this:
con.execute("""
    CREATE OR REPLACE TABLE works AS
    SELECT
        k                                      AS work_id,
        d                                      AS last_modified_date,
        json_extract_string(json_data, '$.title')       AS title,
        json_extract_string(json_data, '$.description') AS description,
        json_extract_string(json_data, '$.created.value') AS created,
        json_extract(json_data, '$.authors')             AS authors_json,
        json_extract(json_data, '$.subjects')            AS subjects_json,
        json_extract(json_data, '$.covers')              AS covers_json,
        json_extract(json_data, '$.type.key')            AS type_key,
        json_data                                        AS raw_json
    FROM works_raw
""")

print(con.execute("SELECT * FROM works LIMIT 5").df())

RuntimeError: Query interrupted